# Mealfit model inference validation

운영 모델 아티팩트가 동일한 입력에 대해 예측 가능한지 검증하는 노트북입니다. 모델 학습 파이프라인은 별도로 관리해야 합니다.

In [ ]:
from pathlib import Path
import re

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

repo_root = Path.cwd()
if not (repo_root / 'services' / 'prediction').exists():
    repo_root = Path.cwd().parents[1]
model_dir = repo_root / 'services' / 'prediction' / 'models'
model_dir

In [ ]:
rf_model = joblib.load(model_dir / 'random_forest_model.pkl')
scaler = joblib.load(model_dir / 'scaler.pkl')
word2idx = joblib.load(model_dir / 'word2idx.pkl')

embedding = nn.Embedding(len(word2idx), 32, padding_idx=0)
embedding.load_state_dict(torch.load(model_dir / 'embedding_layer.pt', map_location='cpu'))
embedding.eval()

In [ ]:
date_string = '2023-10-16'
menu = '흰밥, 근대된장국, 로스팜구이, 두부샐러드, 에너지:326 Kcal, 단백질:12 g'
numeric_features = [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0]

date = pd.to_datetime(date_string)
month, year = date.month, date.year
if month <= 2:
    start, end = pd.to_datetime(f'{year}-01-01'), pd.to_datetime(f'{year}-02-28')
elif month <= 6:
    start, end = pd.to_datetime(f'{year}-03-01'), pd.to_datetime(f'{year}-06-30')
elif month <= 8:
    start, end = pd.to_datetime(f'{year}-07-01'), pd.to_datetime(f'{year}-08-31')
else:
    start, end = pd.to_datetime(f'{year}-09-01'), pd.to_datetime(f'{year}-12-31')
progress = np.clip((date - start).days / (end - start).days, 0, 1)

energy = float(re.search(r'에너지\s*:\s*(\d+(?:\.\d+)?)', menu).group(1))
protein = float(re.search(r'단백질\s*:\s*(\d+(?:\.\d+)?)', menu).group(1))
energy_scaled, protein_scaled, progress_scaled = scaler.transform([[energy, protein, progress]])[0]

indexes = [word2idx.get(token, 0) for token in menu.split()][:20]
indexes += [0] * (20 - len(indexes))
with torch.no_grad():
    menu_average = embedding(torch.tensor(indexes)).mean(dim=0).numpy()

model_input = np.array(numeric_features + [energy_scaled, protein_scaled, progress_scaled] + menu_average.tolist()).reshape(1, -1)
prediction = float(rf_model.predict(model_input)[0])
prediction